# Adaptive Keyframe Retrieval
Mount Drive, install dependencies, set the API key from Colab Secrets, then run one query.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/AICHALLENGENHCM2026/colab_keyframe_agent"
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -y -qq ffmpeg

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/AICHALLENGENHCM2026/colab_keyframe_agent
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [6]:
from keyframe_agent import KeyframeSearchPipeline, load_config
config = load_config('config.yaml')
pipeline = KeyframeSearchPipeline(config)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [79]:
# Chọn 'kis' cho một frame, 'trake' cho E1-E2-..., 'qa' cho 5 gói bằng chứng.
task_type = 'trake'
# Với QA: 'auto' để LLM tự chọn; có thể ép 'single_frame', 'short_window' hoặc 'long_range_temporal'.
qa_mode = 'auto'
query = '''Con lân màu đỏ đang biểu diễn trên các cột trụ.

E1: Con lân treo hai chân trước vào phần dưới thân của hai trụ gần cuối (gần trụ cuối cùng cao nhất).
E2: Con lân đứng thẳng sau khi di chuyển từ cuối dãy trụ về, hai chân đứng trên hai trụ khác nhau và một chân trước co lên.
E3: Khoảnh khắc con lân hoàn tất động tác ngoảnh mặt theo hướng ngược lại.
E4: Khoảnh khắc đầu tiên con lân ngậm một thanh trụ.'''.strip()
result = pipeline.run(query, task_type=task_type, qa_mode=qa_mode)
result

Multi-event CLIP:   0%|          | 0/873 [00:00<?, ?video/s]

{'verified': False,
 'model_verified': False,
 'task_type': 'trake',
 'strategy': 'multi_event_trake',
 'query': 'Con lân màu đỏ đang biểu diễn trên các cột trụ.\n\nE1: Con lân treo hai chân trước vào phần dưới thân của hai trụ gần cuối (gần trụ cuối cùng cao nhất).\nE2: Con lân đứng thẳng sau khi di chuyển từ cuối dãy trụ về, hai chân đứng trên hai trụ khác nhau và một chân trước co lên.\nE3: Khoảnh khắc con lân hoàn tất động tác ngoảnh mặt theo hướng ngược lại.\nE4: Khoảnh khắc đầu tiên con lân ngậm một thanh trụ.',
 'video_id': 'L24_V026',
 'confidence': 0.48,
 'event_coverage': 4,
 'event_count': 4,
 'chronological': True,
 'reason': "The sequence appears in the correct temporal order, but the performer is visibly yellow rather than red, and some event-specific details—especially E1's exact pole positions and E4's first-moment timing—are not fully defensible from the contact sheets.",
 'events': [{'event_id': 'E1',
   'description': 'Con lân treo hai chân trước vào phần dưới thân c

In [78]:
from IPython.display import display, Image, Markdown
if result.get('task_type') == 'qa':
    display(Markdown(f"## Câu hỏi: {result.get('question') or 'Không có'}"))
    display(Markdown(f"**QA mode:** `{result.get('qa_mode', 'single_frame')}` — nguồn: `{result.get('qa_mode_source', 'unknown')}`"))
    display(Markdown(f"**Vật thể/bằng chứng cần tìm:** {result.get('target_object') or 'Không có'}"))
    for rank, bundle in enumerate(result.get('results', []), start=1):
        answer = bundle.get('answer') or 'Chưa có bằng chứng thị giác đủ để trả lời'
        display(Markdown(
            f"## #{rank} — {bundle.get('video_id')} | QA round {bundle.get('qa_round', '-')} | verified={bundle.get('verified', False)} | "
            f"confidence={bundle.get('confidence', 0):.2f}  \
"
            f"**Đáp án:** {answer}  \
"
            f"Nguồn đáp án: {bundle.get('answer_source')} | Đếm: {bundle.get('count_status')}  \
"
            f"Lý do: {bundle.get('reason') or 'Không có'}"
        ))
        answer_ids = {frame.get('candidate_id') for frame in bundle.get('answer_evidence_frames', [])}
        frames = bundle.get('evidence_frames') or []
        if not frames:
            display(Markdown('⚠️ Không có frame bằng chứng hợp lệ.'))
        for frame in frames:
            answer_label = ' | ANSWER FRAME' if frame.get('candidate_id') in answer_ids else ''
            display(Markdown(
                f"frame {frame.get('frame_idx')} | keyframe {frame.get('keyframe_ordinal')} | "
                f"{frame.get('pts_time')}s{answer_label}"
            ))
            path = frame.get('keyframe_path')
            if path:
                display(Image(filename=path))
elif result.get('task_type') == 'trake':
    for event in result.get('events', []):
        display(Markdown(f"## {event['event_id']}: {event.get('description', '')}"))
        selected = event.get('selected')
        alternatives = event.get('alternatives') or []
        items = []
        if selected:
            items.append(selected)
        items.extend(
            item for item in alternatives
            if not selected or item.get('candidate_id') != selected.get('candidate_id')
        )
        if not items:
            display(Markdown(
                f"⚠️ **Không có frame để hiển thị.**  \
"
                f"verified={event.get('verified', False)}  \
"
                f"reason: {event.get('reason', 'Không có lý do')}  \
"
                f"missing: {', '.join(event.get('missing', [])) or 'Không rõ'}"
            ))
            continue
        for rank, item in enumerate(items, start=1):
            label = 'SELECTED' if selected and item.get('candidate_id') == selected.get('candidate_id') else f'ALT #{rank}'
            display(Markdown(
                f"**{label} — {item.get('video_id')} | frame {item.get('frame_idx')} | "
                f"keyframe {item.get('keyframe_ordinal')} | confidence {item.get('confidence', 0):.2f} | "
                f"verified={item.get('verified', False)}**"
            ))
            path = item.get('keyframe_path')
            if path:
                display(Image(filename=path))
else:
    kis_mode = result.get('kis_mode', 'static')
    display(Markdown(f"## KIS mode: `{kis_mode}`"))
    if kis_mode == 'temporal':
        display(Markdown(
            f"Video đủ chi tiết: **{result.get('video_verified', False)}** | "
            f"Frame đại diện hợp lệ: **{result.get('frame_verified', False)}**"
        ))
    keyframes = result.get('keyframes') or ([result] if result.get('keyframe_path') else [])
    for rank, item in enumerate(keyframes, start=1):
        display(Markdown(
            f"**#{rank} — {item.get('video_id')} | frame {item.get('frame_idx')} | "
            f"keyframe {item.get('keyframe_ordinal')} | confidence {item.get('confidence', 0):.2f} | "
            f"verified={item.get('verified', False)}**"
        ))
        path = item.get('keyframe_path')
        if path:
            display(Image(filename=path))

Output hidden; open in https://colab.research.google.com to view.